# Problem 1: RNN, LSTM, GRU on Sequences
Student: Nilesh Ranjan Mitra | ID: 2026aj05496
Hidden size (h): 64 | Seeds: 42, 123, 999

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
import urllib.request
import zipfile

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = "/content/drive/MyDrive/TheRealDNNExp" if IN_COLAB else os.path.abspath("..")
MODULES_DIR = os.path.join(PROJECT_ROOT, "Problem1", "modules")
DATA_ROOT = os.path.join(PROJECT_ROOT, "Data", "har")



if MODULES_DIR not in sys.path:
    sys.path.append(MODULES_DIR)

import numpy as np
import matplotlib.pyplot as plt

from load_har_datasets import load_har, create_subject_split, compute_normalization_stats, normalize_data, HARDataset, build_dataloaders
from plotting import load_activity_labels, plot_class_balance

# Ensure the HAR dataset is available locally, downloading and extracting if necessary. To support Google colab vs local code execution
HAR_ZIP_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"


def ensure_har_dataset(data_root: str) -> str:
    """Download and extract the UCI HAR dataset if not already present.

    Returns the folder containing activity_labels.txt, regardless of
    how many nested folders the zip extracts into.
    """
    for root, _, files in os.walk(data_root):
        if "activity_labels.txt" in files:
            return root

    os.makedirs(data_root, exist_ok=True)
    outer_zip = os.path.join(data_root, "har.zip")
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(HAR_ZIP_URL, outer_zip)

    with zipfile.ZipFile(outer_zip) as zf:
        zf.extractall(data_root)

    # The outer archive may hold a second nested zip; extract that too.
    for root, _, files in os.walk(data_root):
        for name in files:
            if name.endswith(".zip"):
                with zipfile.ZipFile(os.path.join(root, name)) as zf:
                    zf.extractall(root)

    for root, _, files in os.walk(data_root):
        if "activity_labels.txt" in files:
            return root

    raise FileNotFoundError("Could not locate activity_labels.txt after extraction")


DATA_DIR = ensure_har_dataset(DATA_ROOT)
SEEDS = [42, 123, 999]
BATCH_SIZE = 64

In [40]:
# short cuts to load my new .py modlues and few ROOT path w.o. re start the sessions or to survive session restart
import os
import pandas as pd

import importlib
import plotting
import problem1trng

importlib.reload(plotting)
importlib.reload(problem1trng)

from plotting import *
from problem1trng import *

RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "Problem1", "figures")



## Step 2 — Build HAR Data Pipeline (P1.1)
Load raw signals/labels/subjects, verify shapes, build the subject-wise
split, compute normalization statistics, and save class-balance plots.

In [3]:
x_train_full, y_train_full, subjects_train = load_har(DATA_DIR, "train")
x_test, y_test, subjects_test = load_har(DATA_DIR, "test")

print("train:", x_train_full.shape, y_train_full.shape, subjects_train.shape)
print("test: ", x_test.shape, y_test.shape, subjects_test.shape)

Finished stacking HAR signals for split 'train'
Finished loading HAR dataset for split 'train'
Finished stacking HAR signals for split 'test'
Finished loading HAR dataset for split 'test'
train: (7352, 128, 9) (7352,) (7352,)
test:  (2947, 128, 9) (2947,) (2947,)


In [4]:
# Shape checks against the assignment's documented sizes
assert x_train_full.shape == (7352, 128, 9), f"train shape wrong: {x_train_full.shape}"
assert x_test.shape == (2947, 128, 9), f"test shape wrong: {x_test.shape}"
assert y_train_full.shape == (7352,)
assert y_test.shape == (2947,)
assert subjects_train.shape == (7352,)
assert subjects_test.shape == (2947,)

# Label range checks (0-5 after the -1 shift in load_har)
assert y_train_full.min() == 0 and y_train_full.max() == 5
assert y_test.min() == 0 and y_test.max() == 5
assert len(np.unique(y_train_full)) == 6, "expected all 6 classes present in train"
assert len(np.unique(y_test)) == 6, "expected all 6 classes present in test"

# Subject counts and disjointness
unique_train_subjects = np.unique(subjects_train)
unique_test_subjects = np.unique(subjects_test)
assert len(unique_train_subjects) == 21, f"expected 21 train subjects, got {len(unique_train_subjects)}"
assert len(unique_test_subjects) == 9, f"expected 9 test subjects, got {len(unique_test_subjects)}"
assert set(unique_train_subjects).isdisjoint(set(unique_test_subjects)), "train/test subjects overlap!"

# No NaNs/Infs
assert np.isfinite(x_train_full).all(), "NaN/Inf found in train signals"
assert np.isfinite(x_test).all(), "NaN/Inf found in test signals"

print("All HAR shape and label checks passed.")

All HAR shape and label checks passed.


In [5]:
# Per-seed subject split validation
for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    val_subjects = np.unique(subjects_train[val_mask])
    assert len(val_subjects) == 4, f"seed {seed}: expected 4 val subjects, got {len(val_subjects)}"
    assert train_mask.sum() + val_mask.sum() == len(subjects_train)
    assert not (train_mask & val_mask).any(), f"seed {seed}: train/val masks overlap"
    print(f"seed {seed}: train={train_mask.sum()}, val={val_mask.sum()}, val_subjects={val_subjects}")

Finished creating subject split with seed '42'
seed 42: train=5952, val=1400, val_subjects=[ 3 16 22 23]
Finished creating subject split with seed '123'
seed 123: train=5896, val=1456, val_subjects=[ 1  3 19 21]
Finished creating subject split with seed '999'
seed 999: train=5890, val=1462, val_subjects=[ 6 23 28 30]


### Class balance
Plots for the full train set, test set, and the per-seed train/val
subject subsets. Saved under `figures/class_balance/` for review;
final report figures will be selected after Problem 1 is complete.

In [6]:
class_names = load_activity_labels(DATA_DIR)

CLASS_BALANCE_DIR = os.path.join(PROJECT_ROOT, "Problem1", "figures", "class_balance")
os.makedirs(CLASS_BALANCE_DIR, exist_ok=True)

plot_class_balance(
    y_train_full, class_names,
    "Class Balance - Train (Full)",
    f"{CLASS_BALANCE_DIR}/train_full.png",
)
plot_class_balance(
    y_test, class_names,
    "Class Balance - Test",
    f"{CLASS_BALANCE_DIR}/test.png",
)

for seed in SEEDS:
    train_mask, val_mask = create_subject_split(subjects_train, seed)
    plot_class_balance(
        y_train_full[train_mask], class_names,
        f"Class Balance - Train Subset (seed={seed})",
        f"{CLASS_BALANCE_DIR}/train_subset_seed{seed}.png",
    )
    plot_class_balance(
        y_train_full[val_mask], class_names,
        f"Class Balance - Val Subset (seed={seed})",
        f"{CLASS_BALANCE_DIR}/val_subset_seed{seed}.png",
    )

print("Class balance plots saved.")

Finished loading activity labels
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/train_full.png'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/test.png'
Finished creating subject split with seed '42'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/train_subset_seed42.png'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/val_subset_seed42.png'
Finished creating subject split with seed '123'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/train_subset_seed123.png'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Problem1/figures/class_balance/val_subset_seed123.png'
Finished creating subject split with seed '999'
Finished plotting class balance to '/content/drive/MyDrive/TheRealDNNExp/Pro

### Normalization and DataLoaders
Normalization statistics are computed from the training-subject subset
only, for each seed, then applied to that seed's val set and to test.

In [7]:
seed = SEEDS[0]
train_mask, val_mask = create_subject_split(subjects_train, seed)

x_train_subset = x_train_full[train_mask]
y_train_subset = y_train_full[train_mask]
x_val_subset = x_train_full[val_mask]
y_val_subset = y_train_full[val_mask]

mean, std = compute_normalization_stats(x_train_subset)

x_train_norm = normalize_data(x_train_subset, mean, std)
x_val_norm = normalize_data(x_val_subset, mean, std)
x_test_norm = normalize_data(x_test, mean, std)

train_ds = HARDataset(x_train_norm, y_train_subset)
val_ds = HARDataset(x_val_norm, y_val_subset)
test_ds = HARDataset(x_test_norm, y_test)

train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

xb, yb = next(iter(train_loader))
print("batch shapes:", xb.shape, yb.shape)

Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34

## Step 3 — Build Adding-Problem Pipeline (P1.1)
Generate the synthetic adding-problem dataset for T = 50, 100, 200, 400
using the assignment's exact `make_adding` function, and verify
correctness with assert tests.

In [8]:
from load_har_datasets import make_adding, AddingDataset

def verify_adding_data(x: np.ndarray, y: np.ndarray, T: int) -> None:
    """Assert exactly two markers, one per half, and target = sum of marked values."""
    marker_counts = x[:, :, 1].sum(axis=1)
    assert np.all(marker_counts == 2), "every sequence must have exactly two markers"

    first_half_markers = x[:, :T // 2, 1].sum(axis=1)
    second_half_markers = x[:, T // 2:, 1].sum(axis=1)
    assert np.all(first_half_markers == 1), "exactly one marker must be in the first half"
    assert np.all(second_half_markers == 1), "exactly one marker must be in the second half"

    i1 = np.argmax(x[:, :T // 2, 1], axis=1)
    i2 = T // 2 + np.argmax(x[:, T // 2:, 1], axis=1)
    expected_y = x[np.arange(len(x)), i1, 0] + x[np.arange(len(x)), i2, 0]
    assert np.allclose(y, expected_y, atol=1e-6), "target must equal sum of marked values"

In [9]:
T_VALUES = [50, 100, 200, 400]
N_TRAIN, N_VAL, N_TEST = 10_000, 1_000, 1_000
SEED_TRAIN, SEED_VAL, SEED_TEST = 0, 1, 2

# --- Generate the adding-problem data for every T ---
adding_data = {}

for T in T_VALUES:
    x_train_T, y_train_T = make_adding(N_TRAIN, T, SEED_TRAIN)   # renamed
    x_val_T, y_val_T = make_adding(N_VAL, T, SEED_VAL)           # renamed
    x_test_T, y_test_T = make_adding(N_TEST, T, SEED_TEST)       # renamed

    adding_data[T] = {
        "train": (x_train_T, y_train_T),
        "val": (x_val_T, y_val_T),
        "test": (x_test_T, y_test_T),
    }

print(f"T={T}: train={x_train_T.shape}, val={x_val_T.shape}, test={x_test_T.shape}")

# --- Wrap each T's arrays into PyTorch Datasets ---
adding_datasets = {}

for T in T_VALUES:
    x_tr_T, y_tr_T = adding_data[T]["train"]
    x_va_T, y_va_T = adding_data[T]["val"]
    x_te_T, y_te_T = adding_data[T]["test"]
    adding_datasets[T] = {
        "train": AddingDataset(x_tr_T, y_tr_T),
        "val": AddingDataset(x_va_T, y_va_T),
        "test": AddingDataset(x_te_T, y_te_T),
    }

print("Adding-problem datasets wrapped for all T values:", list(adding_datasets.keys()))

# --- Verify correctness of the generated data ---
for T in T_VALUES:
    for split in ["train", "val", "test"]:
        x, y = adding_data[T][split]
        verify_adding_data(x, y, T)
    print(f"T={T}: all checks passed (train/val/test)")

print("All adding-problem data verified successfully.")

Finished generating adding-problem data for T=50, n=10000, seed=0
Finished generating adding-problem data for T=50, n=1000, seed=1
Finished generating adding-problem data for T=50, n=1000, seed=2
Finished generating adding-problem data for T=100, n=10000, seed=0
Finished generating adding-problem data for T=100, n=1000, seed=1
Finished generating adding-problem data for T=100, n=1000, seed=2
Finished generating adding-problem data for T=200, n=10000, seed=0
Finished generating adding-problem data for T=200, n=1000, seed=1
Finished generating adding-problem data for T=200, n=1000, seed=2
Finished generating adding-problem data for T=400, n=10000, seed=0
Finished generating adding-problem data for T=400, n=1000, seed=1
Finished generating adding-problem data for T=400, n=1000, seed=2
T=400: train=(10000, 400, 2), val=(1000, 400, 2), test=(1000, 400, 2)
Adding-problem datasets wrapped for all T values: [50, 100, 200, 400]
T=50: all checks passed (train/val/test)
T=100: all checks passed (

## Step 4 — Implement Custom Cells (P1.2)
Implement MyRNNCell, MyLSTMCell and MyGRUCell from matrix operations,
and verify their forward-pass correctness (shapes, finiteness,
determinism) before comparing them against PyTorch's library layers
in Step 5.

In [10]:
from models import MyRNNCell, MyLSTMCell, MyGRUCell
import torch

D = 9    # HAR channel count
H = 64   # your chosen hidden size
BATCH = 8
torch.manual_seed(0)
x_dummy = torch.randn(BATCH, D)
h0 = torch.zeros(BATCH, H)
c0 = torch.zeros(BATCH, H)

In [11]:
# Test MyRNN forward pass
rnn_cell = MyRNNCell(D, H)
h1 = rnn_cell(x_dummy, h0)

assert h1.shape == (BATCH, H), f"unexpected shape: {h1.shape}"
assert torch.isfinite(h1).all(), "RNN output contains NaN/Inf"

h1_again = rnn_cell(x_dummy, h0)
assert torch.equal(h1, h1_again), "RNN forward pass is not deterministic"

print("MyRNNCell forward pass OK:", h1.shape)

MyRNNCell forward pass OK: torch.Size([8, 64])


In [12]:
# Test MyLSTM forward pass
lstm_cell = MyLSTMCell(D, H)
h1, c1 = lstm_cell(x_dummy, (h0, c0))

assert h1.shape == (BATCH, H) and c1.shape == (BATCH, H), f"unexpected shapes: {h1.shape}, {c1.shape}"
assert torch.isfinite(h1).all() and torch.isfinite(c1).all(), "LSTM output contains NaN/Inf"

h1_again, c1_again = lstm_cell(x_dummy, (h0, c0))
assert torch.equal(h1, h1_again) and torch.equal(c1, c1_again), "LSTM forward pass is not deterministic"

print("MyLSTMCell forward pass OK:", h1.shape, c1.shape)

MyLSTMCell forward pass OK: torch.Size([8, 64]) torch.Size([8, 64])


In [13]:
# Test MyGRU forward pass
gru_cell = MyGRUCell(D, H)
h1 = gru_cell(x_dummy, h0)

assert h1.shape == (BATCH, H), f"unexpected shape: {h1.shape}"
assert torch.isfinite(h1).all(), "GRU output contains NaN/Inf"

h1_again = gru_cell(x_dummy, h0)
assert torch.equal(h1, h1_again), "GRU forward pass is not deterministic"

print("MyGRUCell forward pass OK:", h1.shape)

MyGRUCell forward pass OK: torch.Size([8, 64])


In [14]:
# Count the number of parameters for my chosen hidden size H (which will go in the Report file for reference)
import torch.nn as nn
def count_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())

print("MyRNNCell params:", count_params(rnn_cell))   # expect 4*H + H*D + H*H  = 2H + HD + HH ... see Step 5 derivation
print("MyLSTMCell params:", count_params(lstm_cell))
print("MyGRUCell params:", count_params(gru_cell))

MyRNNCell params: 4800
MyLSTMCell params: 19200
MyGRUCell params: 14400


## Step 5 — Verify Against PyTorch (P1.2)
Copy weights from nn.RNN/nn.LSTM/nn.GRU into the custom cells, run
both over the same random sequence, and confirm the outputs match
within 1e-5. Then derive and reconcile parameter-count formulas.

In [15]:
SEQ_LEN = 20  # arbitrary; any length works since we loop per time step

torch.manual_seed(0)
lib_rnn = nn.RNN(input_size=D, hidden_size=H, num_layers=1, batch_first=True)
lib_lstm = nn.LSTM(input_size=D, hidden_size=H, num_layers=1, batch_first=True)
lib_gru = nn.GRU(input_size=D, hidden_size=H, num_layers=1, batch_first=True)


def copy_weights(custom_cell: nn.Module, library_layer: nn.Module) -> None:
    """Copy layer-0 weights from a PyTorch RNN/LSTM/GRU into a matching custom cell.

    Works for all three because PyTorch names its single-layer parameters
    weight_ih_l0 / weight_hh_l0 / bias_ih_l0 / bias_hh_l0 regardless of
    gate count, and the gate concatenation order already matches ours.
    """
    with torch.no_grad():
        custom_cell.weight_ih.copy_(library_layer.weight_ih_l0)
        custom_cell.weight_hh.copy_(library_layer.weight_hh_l0)
        custom_cell.bias_ih.copy_(library_layer.bias_ih_l0)
        custom_cell.bias_hh.copy_(library_layer.bias_hh_l0)


copy_weights(rnn_cell, lib_rnn)
copy_weights(lstm_cell, lib_lstm)
copy_weights(gru_cell, lib_gru)
print("Weights copied into all three custom cells.")

Weights copied into all three custom cells.


In [16]:
# Comparing over a random sequence for the custom RNN against the library RNN
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in RNN for the same inputs and weights.
x_seq = torch.randn(BATCH, SEQ_LEN, D)  # (batch, seq_len, input_size)

lib_out, _ = lib_rnn(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t = rnn_cell(x_seq[:, t, :], h_t)
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)  # (batch, seq_len, H)

rnn_max_err = (custom_out - lib_out).abs().max().item()
assert rnn_max_err < 1e-5, f"RNN mismatch: max error {rnn_max_err}"
print(f"RNN max abs error: {rnn_max_err:.2e}")

RNN max abs error: 0.00e+00


In [17]:
# Comparing over a random sequence for the custom LSTM against the library LSTM
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in LSTM for the same inputs and weights.
lib_out, _ = lib_lstm(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
c_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t, c_t = lstm_cell(x_seq[:, t, :], (h_t, c_t))
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)

lstm_max_err = (custom_out - lib_out).abs().max().item()
assert lstm_max_err < 1e-5, f"LSTM mismatch: max error {lstm_max_err}"
print(f"LSTM max abs error: {lstm_max_err:.2e}")

LSTM max abs error: 5.96e-08


In [18]:
# Comparing over a random sequence for the custom GRU against the library GRU
# This ensures that our custom implementation produces the same outputs as PyTorch's built-in GRU for the same inputs and weights.
lib_out, _ = lib_gru(x_seq)  # (batch, seq_len, H)

h_t = torch.zeros(BATCH, H)
custom_outputs = []
for t in range(SEQ_LEN):
    h_t = gru_cell(x_seq[:, t, :], h_t)
    custom_outputs.append(h_t)
custom_out = torch.stack(custom_outputs, dim=1)

gru_max_err = (custom_out - lib_out).abs().max().item()
assert gru_max_err < 1e-5, f"GRU mismatch: max error {gru_max_err}"
print(f"GRU max abs error: {gru_max_err:.2e}")

GRU max abs error: 8.94e-08


In [19]:
# Building the verification table for custom RNN, LSTM, and GRU cells against PyTorch's built-in implementations
# This table will summarize the number of parameters (both actual and minimal) and the maximum absolute error for each custom cell.
# Also this will server as evidence in the final report
import pandas as pd

def actual_params(gates: int, d: int, h: int) -> int:
    return gates * (h * d + h * h + 2 * h)

def minimal_params(gates: int, d: int, h: int) -> int:
    return gates * (h * d + h * h + h)

gate_counts = {"RNN": 1, "LSTM": 4, "GRU": 3}
max_errors = {"RNN": rnn_max_err, "LSTM": lstm_max_err, "GRU": gru_max_err}
custom_cells = {"RNN": rnn_cell, "LSTM": lstm_cell, "GRU": gru_cell}

rows = []
for name, gates in gate_counts.items():
    rows.append({
        "Model": name,
        "Gates": gates,
        "Actual params (formula)": actual_params(gates, D, H),
        "Actual params (code)": count_params(custom_cells[name]),
        "Minimal params (hand-derived)": minimal_params(gates, D, H),
        "Redundant bias params": gates * H,
        "Max abs error vs PyTorch": f"{max_errors[name]:.2e}",
    })

verification_table = pd.DataFrame(rows)
verification_table

,Model,Gates,Actual params (formula),Actual params (code),Minimal params (hand-derived),Redundant bias params,Max abs error vs PyTorch
0,RNN,1,4800,4800,4736,64,0.00e+00
1,LSTM,4,19200,19200,18944,256,5.96e-08
2,GRU,3,14400,14400,14208,192,8.94e-08


## Step 6 — HAR Model Training (P1.3)
Train RNN, LSTM and GRU classifiers (library layers) on HAR with
identical hidden size, optimizer, batch size and epoch budget, across
3 seeds each. Save results.csv rows, checkpoints, and confusion
matrices for all 9 runs.

In [27]:
from models import HARClassifier
from problem1trng import train_model, evaluate, compute_metrics, append_result_row
from plotting import plot_confusion_matrix

MODEL_TYPES = ["rnn", "lstm", "gru"]
EPOCHS = 30  # same budget for all three models; adjust and note in report
NUM_CLASSES = 6

CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "Problem1", "checkpoints")
CONF_MATRIX_DIR = os.path.join(PROJECT_ROOT, "Problem1", "figures", "confusion_matrix")
RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(CONF_MATRIX_DIR, exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_CSV_PATH), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [28]:
run_results = []

for model_type in MODEL_TYPES:
    for seed in SEEDS:
        print(f"\n=== Training {model_type.upper()} | seed={seed} ===")
        torch.manual_seed(seed)

        train_mask, val_mask = create_subject_split(subjects_train, seed)
        x_tr, y_tr = x_train_full[train_mask], y_train_full[train_mask]
        x_va, y_va = x_train_full[val_mask], y_train_full[val_mask]

        mean, std = compute_normalization_stats(x_tr)
        x_tr_norm = normalize_data(x_tr, mean, std)
        x_va_norm = normalize_data(x_va, mean, std)
        x_te_norm = normalize_data(x_test, mean, std)

        train_ds = HARDataset(x_tr_norm, y_tr)
        val_ds = HARDataset(x_va_norm, y_va)
        test_ds = HARDataset(x_te_norm, y_test)
        train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

        model = HARClassifier(model_type, input_size=D, hidden_size=H, num_classes=NUM_CLASSES).to(device)
        sec_per_epoch = train_model(model, train_loader, val_loader, EPOCHS, lr=1e-3, device=device)

        y_true, y_pred = evaluate(model, test_loader, device)
        metrics = compute_metrics(y_true, y_pred)
        params = sum(p.numel() for p in model.parameters())

        checkpoint_path = os.path.join(CHECKPOINTS_DIR, f"{model_type}_seed{seed}.pt")
        torch.save(model.state_dict(), checkpoint_path)

        cm_path = os.path.join(CONF_MATRIX_DIR, f"{model_type}_seed{seed}.png")
        plot_confusion_matrix(
            y_true, y_pred, class_names,
            f"Confusion Matrix - {model_type.upper()} (seed={seed})",
            cm_path,
        )

        row = {
            "problem": "P1.3",
            "model": model_type,
            "seed": seed,
            "seq_len": 128,
            "test_acc": metrics["test_acc"],
            "macro_f1": metrics["macro_f1"],
            "params": params,
            "sec_per_epoch": sec_per_epoch,
            "notes": f"HAR classification, h={H}",
        }
        append_result_row(RESULTS_CSV_PATH, row)
        run_results.append(row)

        print(f"  test_acc={metrics['test_acc']:.4f}  macro_f1={metrics['macro_f1']:.4f}  "
              f"params={params}  sec/epoch={sec_per_epoch:.1f}  checkpoint={checkpoint_path}")

print("\nAll 9 HAR training runs complete.")


=== Training RNN | seed=42 ===
Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.259813

In [29]:
best_run = max(run_results, key=lambda r: r["macro_f1"])
print(f"Best model by macro-F1: {best_run['model'].upper()} (seed={best_run['seed']})")
print(f"  test_acc={best_run['test_acc']:.4f}  macro_f1={best_run['macro_f1']:.4f}")
print(f"  Confusion matrix: figures/confusion_matrix/{best_run['model']}_seed{best_run['seed']}.png")
print("  Note this run as the 'best model' in report.pdf per the macro-F1 criterion.")

Best model by macro-F1: GRU (seed=999)
  test_acc=0.9226  macro_f1=0.9228
  Confusion matrix: figures/confusion_matrix/gru_seed999.png
  Note this run as the 'best model' in report.pdf per the macro-F1 criterion.


## Step 7 — Adding Problem Experiments (P1.4)
Train RNN, LSTM and GRU on the adding problem for T = 50, 100, 200, 400,
3 seeds each (36 runs). Save results.csv rows and checkpoints for every
run; capture a learning-curve history at T=200 (seed 42); build the
MSE-vs-T plot and the solved-run table.

In [30]:
from models import AddingRegressor
from problem1trng import train_regression_model, evaluate_regression, compute_mse
from plotting import plot_learning_curves, plot_mse_vs_t
import pandas as pd

EPOCHS_BY_T = {50: 60, 100: 80, 200: 120, 400: 150}
INPUT_SIZE_ADDING = 2
LEARNING_CURVE_SEED = 42
LEARNING_CURVE_T = 200

ADDING_CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "Problem1", "checkpoints", "adding")
os.makedirs(ADDING_CHECKPOINTS_DIR, exist_ok=True)

In [31]:
# Main loop: 4 T values × 3 models × 3 seeds
adding_run_results = []
learning_curve_histories = {}

for T in T_VALUES:
    epochs = EPOCHS_BY_T[T]
    for model_type in MODEL_TYPES:
        for seed in SEEDS:
            print(f"\n=== Adding Problem | {model_type.upper()} | T={T} | seed={seed} ===")
            torch.manual_seed(seed)

            train_loader, val_loader, test_loader = build_dataloaders(
                adding_datasets[T]["train"],
                adding_datasets[T]["val"],
                adding_datasets[T]["test"],
                BATCH_SIZE,
            )

            model = AddingRegressor(model_type, input_size=INPUT_SIZE_ADDING, hidden_size=H).to(device)

            track_history = (T == LEARNING_CURVE_T and seed == LEARNING_CURVE_SEED)
            sec_per_epoch, history = train_regression_model(
                model, train_loader, val_loader, epochs, lr=1e-3, device=device,
                track_history=track_history,
            )
            if track_history:
                learning_curve_histories[model_type] = history

            y_true, y_pred = evaluate_regression(model, test_loader, device)
            test_mse = compute_mse(y_true, y_pred)
            params = sum(p.numel() for p in model.parameters())

            checkpoint_path = os.path.join(ADDING_CHECKPOINTS_DIR, f"{model_type}_T{T}_seed{seed}.pt")
            torch.save(model.state_dict(), checkpoint_path)

            row = {
                "problem": "P1.4",
                "model": model_type,
                "seed": seed,
                "seq_len": T,
                "test_mse": test_mse,
                "params": params,
                "sec_per_epoch": sec_per_epoch,
                "notes": f"Adding problem, h={H}, epochs={epochs}",
            }
            append_result_row(RESULTS_CSV_PATH, row)
            adding_run_results.append(row)

            print(f"  test_mse={test_mse:.4f}  params={params}  sec/epoch={sec_per_epoch:.1f}  checkpoint={checkpoint_path}")

print("\nAll 36 adding-problem training runs complete.")


=== Adding Problem | RNN | T=50 | seed=42 ===
Finished building dataloaders
  epoch 10/60 - val_mse=0.1656 - 0.4s
  epoch 20/60 - val_mse=0.1621 - 0.4s
  epoch 30/60 - val_mse=0.1666 - 0.4s
  epoch 40/60 - val_mse=0.1643 - 0.4s
  epoch 50/60 - val_mse=0.1658 - 0.4s
  epoch 60/60 - val_mse=0.1603 - 0.4s
  test_mse=0.1693  params=4417  sec/epoch=0.4  checkpoint=/content/drive/MyDrive/TheRealDNNExp/Problem1/checkpoints/adding/rnn_T50_seed42.pt

=== Adding Problem | RNN | T=50 | seed=123 ===
Finished building dataloaders
  epoch 10/60 - val_mse=0.1653 - 0.5s
  epoch 20/60 - val_mse=0.1654 - 0.4s
  epoch 30/60 - val_mse=0.1652 - 0.4s
  epoch 40/60 - val_mse=0.0485 - 0.4s
  epoch 50/60 - val_mse=0.0165 - 0.4s
  epoch 60/60 - val_mse=0.0142 - 0.4s
  test_mse=0.0144  params=4417  sec/epoch=0.4  checkpoint=/content/drive/MyDrive/TheRealDNNExp/Problem1/checkpoints/adding/rnn_T50_seed123.pt

=== Adding Problem | RNN | T=50 | seed=999 ===
Finished building dataloaders
  epoch 10/60 - val_mse=0.16

In [32]:
# MSE summary table (mean ± std across seeds)

adding_df = pd.DataFrame(adding_run_results)

summary_df = (
    adding_df.groupby(["model", "seq_len"])["test_mse"]
    .agg(mse_mean="mean", mse_std="std")
    .reset_index()
)
print("MSE summary (mean ± std across 3 seeds):")
summary_df

MSE summary (mean ± std across 3 seeds):


,model,seq_len,mse_mean,mse_std
0,gru,50,0.000116,0.000075
1,gru,100,0.000466,0.000408
2,gru,200,0.000058,0.000021
3,gru,400,0.000076,0.000047
4,lstm,50,0.000602,0.000254
5,lstm,100,0.000301,0.000056
6,lstm,200,0.001001,0.001117
7,lstm,400,0.102624,0.083191
8,rnn,50,0.067876,0.087909
9,rnn,100,0.167416,0.002301


In [33]:
# Solved-run table (MSE < 0.05)
adding_df["solved"] = adding_df["test_mse"] < 0.05
solved_table = (
    adding_df.groupby(["model", "seq_len"])["solved"]
    .sum()
    .reset_index()
    .pivot(index="model", columns="seq_len", values="solved")
)
print("Solved-run table (out of 3 seeds, MSE < 0.05):")
solved_table

Solved-run table (out of 3 seeds, MSE < 0.05):


seq_len,50,100,200,400
model,,,,
gru,3,3,3,3
lstm,3,3,3,1
rnn,2,0,0,0


In [34]:
FIGURES_DIR = os.path.join(PROJECT_ROOT, "Problem1", "figures")

mse_vs_t_path = os.path.join(FIGURES_DIR, "mse_vs_T.png")
plot_mse_vs_t(summary_df, "Test MSE vs Sequence Length T", mse_vs_t_path)

learning_curves_path = os.path.join(FIGURES_DIR, f"learning_curves_T{LEARNING_CURVE_T}.png")
plot_learning_curves(
    learning_curve_histories,
    f"Learning Curves at T={LEARNING_CURVE_T} (seed={LEARNING_CURVE_SEED})",
    learning_curves_path,
)

print("Plots saved:", mse_vs_t_path, learning_curves_path)

Plots saved: /content/drive/MyDrive/TheRealDNNExp/Problem1/figures/mse_vs_T.png /content/drive/MyDrive/TheRealDNNExp/Problem1/figures/learning_curves_T200.png


## Step 8 — Gradient Analysis (P1.5a)
At T=100, capture ||dL/dh_t|| for each of RNN/LSTM/GRU before training
(random init) and after training (reusing Step 7's T=100, seed=42
checkpoints). A fixed batch of 256 test sequences is used for all six
cases (documented assumption — batch size not specified by the rubric).

In [35]:
from problem1trng import compute_gradient_norms
from plotting import plot_gradient_norms

GRADIENT_T = 100
GRADIENT_SEED = 42
GRADIENT_BATCH_SIZE = 256

# Fixed batch of T=100 test sequences, reused across all six (model x before/after) cases
x_grad_full, y_grad_full = adding_data[GRADIENT_T]["test"]
x_grad_batch = torch.from_numpy(x_grad_full[:GRADIENT_BATCH_SIZE]).float()
y_grad_batch = torch.from_numpy(y_grad_full[:GRADIENT_BATCH_SIZE]).float()
print("Gradient-analysis batch shape:", x_grad_batch.shape, y_grad_batch.shape)

print(torch.is_grad_enabled())

Gradient-analysis batch shape: torch.Size([256, 100, 2]) torch.Size([256])
True


In [36]:
# Compute before/after gradient norms for all three models
before_norms = {}
after_norms = {}

for model_type in MODEL_TYPES:
    # Before training: freshly initialized model, no checkpoint loaded
    torch.manual_seed(GRADIENT_SEED)
    model_before = AddingRegressor(model_type, input_size=INPUT_SIZE_ADDING, hidden_size=H).to(device)
    before_norms[model_type] = compute_gradient_norms(model_before, x_grad_batch, y_grad_batch, device)

    # After training: reuse the Step 7 checkpoint trained at T=100, seed=42
    model_after = AddingRegressor(model_type, input_size=INPUT_SIZE_ADDING, hidden_size=H).to(device)
    checkpoint_path = os.path.join(ADDING_CHECKPOINTS_DIR, f"{model_type}_T{GRADIENT_T}_seed{GRADIENT_SEED}.pt")
    model_after.load_state_dict(torch.load(checkpoint_path, map_location=device))
    after_norms[model_type] = compute_gradient_norms(model_after, x_grad_batch, y_grad_batch, device)

    print(f"{model_type.upper()}: before[t=1]={before_norms[model_type][0]:.2e}, "
          f"before[t={GRADIENT_T}]={before_norms[model_type][-1]:.2e}, "
          f"after[t=1]={after_norms[model_type][0]:.2e}, "
          f"after[t={GRADIENT_T}]={after_norms[model_type][-1]:.2e}")

RNN: before[t=1]=0.00e+00, before[t=100]=3.75e-03, after[t=1]=0.00e+00, after[t=100]=7.77e-04
LSTM: before[t=1]=0.00e+00, before[t=100]=5.01e-03, after[t=1]=0.00e+00, after[t=100]=1.06e-04
GRU: before[t=1]=0.00e+00, before[t=100]=4.11e-03, after[t=1]=0.00e+00, after[t=100]=2.07e-04


In [37]:
# Generate the combined comparison figure
gradient_plot_path = os.path.join(FIGURES_DIR, "gradient_norms_T100.png")
plot_gradient_norms(
    before_norms, after_norms,
    f"Gradient Norm ||dL/dh_t|| at T={GRADIENT_T} (batch={GRADIENT_BATCH_SIZE}, seed={GRADIENT_SEED})",
    gradient_plot_path,
)
print("Gradient norm plot saved:", gradient_plot_path)

Gradient norm plot saved: /content/drive/MyDrive/TheRealDNNExp/Problem1/figures/gradient_norms_T100.png


## Step 9a — Hidden-Size Sweep (P1.5b)
Train LSTM and GRU on HAR at h = 16, 32, 128 (3 seeds each, 18 new runs).
h=64 is reused directly from Step 6's existing results.

In [38]:
# config and main sweep loop

HIDDEN_SIZES_SWEEP = [16, 32, 128]  # h=64 already covered by Step 6
SWEEP_MODEL_TYPES = ["lstm", "gru"]

SWEEP_CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "Problem1", "checkpoints", "hidden_sweep")
os.makedirs(SWEEP_CHECKPOINTS_DIR, exist_ok=True)

sweep_run_results = []

for hidden_size in HIDDEN_SIZES_SWEEP:
    for model_type in SWEEP_MODEL_TYPES:
        for seed in SEEDS:
            print(f"\n=== Hidden-size sweep | {model_type.upper()} | h={hidden_size} | seed={seed} ===")
            torch.manual_seed(seed)

            train_mask, val_mask = create_subject_split(subjects_train, seed)
            x_tr, y_tr = x_train_full[train_mask], y_train_full[train_mask]
            x_va, y_va = x_train_full[val_mask], y_train_full[val_mask]

            mean, std = compute_normalization_stats(x_tr)
            x_tr_norm = normalize_data(x_tr, mean, std)
            x_va_norm = normalize_data(x_va, mean, std)
            x_te_norm = normalize_data(x_test, mean, std)

            train_ds = HARDataset(x_tr_norm, y_tr)
            val_ds = HARDataset(x_va_norm, y_va)
            test_ds = HARDataset(x_te_norm, y_test)
            train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

            model = HARClassifier(model_type, input_size=D, hidden_size=hidden_size, num_classes=NUM_CLASSES).to(device)
            sec_per_epoch = train_model(model, train_loader, val_loader, EPOCHS, lr=1e-3, device=device)

            y_true, y_pred = evaluate(model, test_loader, device)
            metrics = compute_metrics(y_true, y_pred)
            params = sum(p.numel() for p in model.parameters())

            checkpoint_path = os.path.join(SWEEP_CHECKPOINTS_DIR, f"{model_type}_h{hidden_size}_seed{seed}.pt")
            torch.save(model.state_dict(), checkpoint_path)

            row = {
                "problem": "P1.5b",
                "model": model_type,
                "seed": seed,
                "seq_len": 128,
                "test_acc": metrics["test_acc"],
                "macro_f1": metrics["macro_f1"],
                "params": params,
                "sec_per_epoch": sec_per_epoch,
                "notes": f"Hidden-size sweep, h={hidden_size}",
            }
            append_result_row(RESULTS_CSV_PATH, row)
            row["hidden_size"] = hidden_size  # kept in memory only, not written to results.csv
            sweep_run_results.append(row)

            print(f"  test_acc={metrics['test_acc']:.4f}  macro_f1={metrics['macro_f1']:.4f}  params={params}  sec/epoch={sec_per_epoch:.1f}")

print("\nAll 18 hidden-size sweep runs complete.")


=== Hidden-size sweep | LSTM | h=16 | seed=42 ===
Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  

In [41]:
sweep_df = pd.DataFrame(sweep_run_results)

# Reuse Step 6's existing h=64 LSTM/GRU results (already in memory as run_results)
h64_records = [{**r, "hidden_size": H} for r in run_results if r["model"] in SWEEP_MODEL_TYPES]
h64_df = pd.DataFrame(h64_records)

full_sweep_df = pd.concat([sweep_df, h64_df], ignore_index=True)

sweep_summary = (
    full_sweep_df.groupby(["model", "hidden_size"])["macro_f1"]
    .agg(f1_mean="mean", f1_std="std")
    .reset_index()
)
print(sweep_summary)

sweep_plot_path = os.path.join(FIGURES_DIR, "hidden_size_sweep.png")
plot_hidden_size_sweep(sweep_summary, "Hidden-Size Sweep (LSTM, GRU) on HAR", sweep_plot_path)
print("Hidden-size sweep plot saved:", sweep_plot_path)

  model  hidden_size   f1_mean    f1_std
0   gru           16  0.877512  0.002448
1   gru           32  0.893138  0.011325
2   gru           64  0.909314  0.011680
3   gru          128  0.904508  0.009616
4  lstm           16  0.856483  0.015255
5  lstm           32  0.828306  0.082531
6  lstm           64  0.893391  0.013304
7  lstm          128  0.890775  0.007586
Hidden-size sweep plot saved: /content/drive/MyDrive/TheRealDNNExp/Problem1/figures/hidden_size_sweep.png


## Step 9b — Gradient-Clipping Ablation (P1.5c)
Vanilla RNN, T=200, gradient clipping (norm 1.0) vs. none. The
"no clipping" condition reuses Step 7's existing RNN@T=200 runs and
its seed-42 learning-curve history; only the "clipped" condition is
newly trained (3 seeds).

In [42]:
ABLATION_T = 200
ABLATION_SEED_FOR_CURVE = 42
ABLATION_CLIP_NORM = 1.0

ablation_run_results = []
ablation_histories = {}

# Reuse Step 7's existing "no clipping" RNN @ T=200 runs (3 seeds, already trained)
no_clip_rows = [r for r in adding_run_results if r["model"] == "rnn" and r["seq_len"] == ABLATION_T]
assert len(no_clip_rows) == 3, f"expected 3 existing no-clip RNN@T={ABLATION_T} runs, found {len(no_clip_rows)}"
for r in no_clip_rows:
    ablation_run_results.append({**r, "clip": "none"})

ablation_histories["none"] = learning_curve_histories["rnn"]  # already captured in Step 7

# Train 3 NEW RNN@T=200 runs WITH gradient clipping
for seed in SEEDS:
    print(f"\n=== Gradient-clipping ablation | RNN | T={ABLATION_T} | clip={ABLATION_CLIP_NORM} | seed={seed} ===")
    torch.manual_seed(seed)

    train_loader, val_loader, test_loader = build_dataloaders(
        adding_datasets[ABLATION_T]["train"],
        adding_datasets[ABLATION_T]["val"],
        adding_datasets[ABLATION_T]["test"],
        BATCH_SIZE,
    )

    model = AddingRegressor("rnn", input_size=INPUT_SIZE_ADDING, hidden_size=H).to(device)

    track_history = (seed == ABLATION_SEED_FOR_CURVE)
    sec_per_epoch, history = train_regression_model(
        model, train_loader, val_loader, EPOCHS_BY_T[ABLATION_T], lr=1e-3, device=device,
        track_history=track_history, max_grad_norm=ABLATION_CLIP_NORM,
    )
    if track_history:
        ablation_histories["clipped"] = history

    y_true, y_pred = evaluate_regression(model, test_loader, device)
    test_mse = compute_mse(y_true, y_pred)
    params = sum(p.numel() for p in model.parameters())

    checkpoint_path = os.path.join(ADDING_CHECKPOINTS_DIR, f"rnn_T{ABLATION_T}_clipped_seed{seed}.pt")
    torch.save(model.state_dict(), checkpoint_path)

    row = {
        "problem": "P1.5c",
        "model": "rnn",
        "seed": seed,
        "seq_len": ABLATION_T,
        "test_mse": test_mse,
        "params": params,
        "sec_per_epoch": sec_per_epoch,
        "notes": f"Gradient-clipping ablation (clip_norm={ABLATION_CLIP_NORM}), h={H}",
    }
    append_result_row(RESULTS_CSV_PATH, row)
    ablation_run_results.append({**row, "clip": "clipped"})

    print(f"  test_mse={test_mse:.4f}  params={params}  sec/epoch={sec_per_epoch:.1f}")

print("\nGradient-clipping ablation training complete.")


=== Gradient-clipping ablation | RNN | T=200 | clip=1.0 | seed=42 ===
Finished building dataloaders
  epoch 10/120 - val_mse=0.1603 - 0.5s
  epoch 20/120 - val_mse=0.1597 - 0.5s
  epoch 30/120 - val_mse=0.1607 - 0.5s
  epoch 40/120 - val_mse=0.1598 - 0.6s
  epoch 50/120 - val_mse=0.1600 - 0.4s
  epoch 60/120 - val_mse=0.1605 - 0.5s
  epoch 70/120 - val_mse=0.1605 - 0.6s
  epoch 80/120 - val_mse=0.1607 - 0.5s
  epoch 90/120 - val_mse=0.1615 - 0.4s
  epoch 100/120 - val_mse=0.1603 - 0.6s
  epoch 110/120 - val_mse=0.1631 - 0.4s
  epoch 120/120 - val_mse=0.1599 - 0.4s
  test_mse=0.1685  params=4417  sec/epoch=0.5

=== Gradient-clipping ablation | RNN | T=200 | clip=1.0 | seed=123 ===
Finished building dataloaders
  epoch 10/120 - val_mse=0.1598 - 0.5s
  epoch 20/120 - val_mse=0.1600 - 0.4s
  epoch 30/120 - val_mse=0.1599 - 0.4s
  epoch 40/120 - val_mse=0.1603 - 0.5s
  epoch 50/120 - val_mse=0.1599 - 0.5s
  epoch 60/120 - val_mse=0.1600 - 0.4s
  epoch 70/120 - val_mse=0.1599 - 0.4s
  epoch

In [43]:
# Build the comparison figure
ablation_df = pd.DataFrame(ablation_run_results)
ablation_summary = (
    ablation_df.groupby("clip")["test_mse"]
    .agg(mse_mean="mean", mse_std="std")
    .reset_index()
)
print(ablation_summary)

ablation_plot_path = os.path.join(FIGURES_DIR, "gradient_clipping_ablation_T200.png")
plot_gradient_clipping_ablation(
    ablation_histories, ablation_summary,
    f"Gradient Clipping Ablation - Vanilla RNN at T={ABLATION_T}",
    ablation_plot_path,
)
print("Ablation plot saved:", ablation_plot_path)

      clip  mse_mean   mse_std
0  clipped  0.173034  0.008230
1     none  0.170774  0.004642
Ablation plot saved: /content/drive/MyDrive/TheRealDNNExp/Problem1/figures/gradient_clipping_ablation_T200.png


# HAR summary table (P1.3, your base h=64 comparison)

In [44]:
results_df = pd.read_csv(RESULTS_CSV_PATH)
print("Total rows:", len(results_df))
print(results_df["problem"].value_counts())
har_df = results_df[results_df["problem"] == "P1.3"].copy()
assert len(har_df) == 9, f"expected 9 HAR rows, found {len(har_df)}"

har_summary = (
    har_df.groupby("model")
    .agg(
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        params=("params", "first"),
        sec_per_epoch_mean=("sec_per_epoch", "mean"),
    )
    .reset_index()
)
print("HAR Summary Table (h=64, 3 seeds):")
har_summary

Total rows: 66
problem
P1.4     36
P1.5b    18
P1.3      9
P1.5c     3
Name: count, dtype: int64
HAR Summary Table (h=64, 3 seeds):


,model,test_acc_mean,test_acc_std,macro_f1_mean,macro_f1_std,params,sec_per_epoch_mean
0,gru,0.908947,0.011854,0.909314,0.011680,14790,0.258597
1,lstm,0.893903,0.011383,0.893391,0.013304,19590,0.274165
2,rnn,0.728198,0.057876,0.722430,0.062962,5190,0.254153


# Adding-Problem summary table (P1.4, all T values)

In [45]:
adding_df = results_df[results_df["problem"] == "P1.4"].copy()
assert len(adding_df) == 36, f"expected 36 adding-problem rows, found {len(adding_df)}"

adding_summary = (
    adding_df.groupby(["model", "seq_len"])
    .agg(
        test_mse_mean=("test_mse", "mean"),
        test_mse_std=("test_mse", "std"),
        params=("params", "first"),
        sec_per_epoch_mean=("sec_per_epoch", "mean"),
    )
    .reset_index()
)

adding_df["solved"] = adding_df["test_mse"] < 0.05
solved_counts = (
    adding_df.groupby(["model", "seq_len"])["solved"]
    .sum()
    .reset_index()
    .rename(columns={"solved": "num_solved_of_3"})
)

adding_summary = adding_summary.merge(solved_counts, on=["model", "seq_len"])
print("Adding-Problem Summary Table (3 seeds per T):")
adding_summary

Adding-Problem Summary Table (3 seeds per T):


,model,seq_len,test_mse_mean,test_mse_std,params,sec_per_epoch_mean,num_solved_of_3
0,gru,50,0.000116,0.000075,13121,0.409783,3
1,gru,100,0.000466,0.000408,13121,0.418901,3
2,gru,200,0.000058,0.000021,13121,0.430654,3
3,gru,400,0.000076,0.000047,13121,0.516236,3
4,lstm,50,0.000602,0.000254,17473,0.425628,3
5,lstm,100,0.000301,0.000056,17473,0.440532,3
6,lstm,200,0.001001,0.001117,17473,0.449450,3
7,lstm,400,0.102624,0.083191,17473,0.490148,1
8,rnn,50,0.067876,0.087909,4417,0.391259,2
9,rnn,100,0.167416,0.002301,4417,0.388945,0
